# Upscale Your Own Low-Resolution Image

Upload a low-resolution photo from your computer and enhance it with Real-ESRGAN. You can pick the output scale (2x, 3x or 4x), compare a zoomed crop against a normal bicubic resize, and download the result.

## How to run
- Open in Google Colab. Runtime type: **T4 GPU**.
- Run the cells from top to bottom.

_Rewritten from my original Colab notebook: `Upscale low resolutions image.ipynb`._

In [ ]:
!pip install -q realesrgan

## Fix a torchvision compatibility issue
`basicsr` (used by Real-ESRGAN) still imports `torchvision.transforms.functional_tensor`, which newer torchvision removed. This cell patches that one import line.

In [ ]:
import importlib.util
from pathlib import Path

# find_spec locates the package without importing it (importing would fail before the patch)
basicsr_dir = Path(importlib.util.find_spec("basicsr").origin).parent
deg_file = basicsr_dir / "data" / "degradations.py"
code = deg_file.read_text()
code = code.replace("torchvision.transforms.functional_tensor", "torchvision.transforms.functional")
deg_file.write_text(code)
print("Patched:", deg_file)

## Load the Real-ESRGAN model
The weights download automatically the first time.

In [ ]:
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

# RealESRGAN_x4plus: a general-purpose 4x upscaling model
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
    model=model,
    tile=400,          # process the image in tiles so big images fit in GPU memory
    tile_pad=10,
    pre_pad=0,
    half=torch.cuda.is_available(),  # half precision is faster on GPU
)
print("Model ready")

In [ ]:
import numpy as np
from PIL import Image

def upscale(pil_image, outscale=4):
    """Upscale a PIL image with Real-ESRGAN and return a PIL image."""
    rgb = np.array(pil_image.convert("RGB"))
    bgr = rgb[:, :, ::-1]                      # Real-ESRGAN expects BGR (OpenCV order)
    out_bgr, _ = upsampler.enhance(bgr, outscale=outscale)
    return Image.fromarray(out_bgr[:, :, ::-1])

## Upload an image

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded))
original = Image.open(filename).convert("RGB")
print(filename, original.size)

## Upscale
Change `SCALE` to 2, 3 or 4.

In [ ]:
SCALE = 4
upscaled = upscale(original, outscale=SCALE)
print("New size:", upscaled.size)

## Compare a zoomed-in crop
The original crop is resized with simple bicubic interpolation so both crops have the same size.

In [ ]:
import matplotlib.pyplot as plt

w, h = original.size
box = (w // 4, h // 4, w // 2, h // 2)   # a small region of the image
crop_orig = original.crop(box).resize(((box[2] - box[0]) * SCALE, (box[3] - box[1]) * SCALE), Image.BICUBIC)
crop_up = upscaled.crop(tuple(v * SCALE for v in box))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(crop_orig)
axes[0].set_title("Bicubic resize")
axes[1].imshow(crop_up)
axes[1].set_title("Real-ESRGAN")
for ax in axes:
    ax.axis("off")
plt.show()

## Download the result

In [ ]:
out_name = "upscaled_" + filename.rsplit(".", 1)[0] + ".png"
upscaled.save(out_name)
files.download(out_name)